# 08 — Long-vol breakout event study (15m / 1h Donchian + VWAP)

Research-only. Uses frozen `configs/long_vol_v1.yaml`. **Not** the live TEMA scanner.

1. Load Binance Vision **15m** USDT-M klines (cached under `backtest/data/cache`).
2. Build features (`long_vol.features`) with Pine-parity ATR/RSI/ADX.
3. Log every **compression + 1h Donchian break** trigger; forward returns from **next bar open**.
4. Gate attribution (which filter blocks edge).
5. Matplotlib PNGs → `research/notebooks/output/long_vol/`.

Run headless: `python research/notebooks/run_long_vol_jupyter.py` from `python/`.

In [ ]:
import sys
from datetime import date, timedelta
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "python" / "long_vol").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root:", ROOT)

In [ ]:
from IPython.display import Image, display
import pandas as pd
from backtest.data_loader import load_klines
from long_vol.config import load_config
from long_vol.backtest.event_study import run_event_study, bucket_summary
from long_vol.viz import (
    save_horizon_means,
    save_return_hist,
    save_gate_funnel,
    save_bucket_bars,
    save_equity_curve,
    save_feature_scatter,
)

SYMBOLS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "LINKUSDT", "AVAXUSDT", "DOGEUSDT", "BNBUSDT"]
LOOKBACK_DAYS = 365
OUT = Path("output/long_vol")
OUT.mkdir(parents=True, exist_ok=True)

cfg = load_config(ROOT / "configs" / "long_vol_v1.yaml")
end = date.today() - timedelta(days=1)
start = end - timedelta(days=LOOKBACK_DAYS)
panels = {}
for sym in SYMBOLS:
    df = load_klines(sym, "15m", start, end)
    if len(df) > 500:
        panels[sym] = df
        print(sym, len(df), df.index[0].date(), "→", df.index[-1].date())

In [ ]:
result = run_event_study(panels, cfg, require_compression=True)
ev = result.events
print("raw triggers:", len(ev), "| all gates:", int(ev["all_gates"].sum()) if len(ev) else 0)
display(result.attribution)
ev.head()

In [ ]:
save_horizon_means(ev, OUT / "horizon_means.png", title=f"Mean FWD return ({start} → {end})")
save_return_hist(ev, OUT / "ret_24h_hist.png")
save_gate_funnel(result.attribution, OUT / "gate_funnel.png")
save_feature_scatter(ev, OUT / "scatter_dist_atr.png")
save_bucket_bars(bucket_summary(ev, "dist_atr"), OUT / "bucket_dist_atr.png")
save_equity_curve(ev, OUT / "equity_nonoverlap_24h.png")

for p in sorted(OUT.glob("*.png")):
    display(Image(filename=str(p)))

In [ ]:
cols = [c for c in ev.columns if c.startswith("ret_") and c.endswith("_pct")]
summary = ev[cols].describe().T[["mean", "50%", "count"]]
summary.columns = ["mean_pct", "median_pct", "n"]
summary